# Parte 1 — Ingestão e Preparação (Camadas Bronze e Silver)

###  1.1. Preparação e Upload dos Dados
Carregue os arquivos CSV disponibilizados na pasta Inputs como Volume no Data Lakehouse do Databricks.


In [0]:
# Importações e criação da sessão Spark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

#Criar sessão spark
spark = SparkSession.builder.appName("AtividadePraticaMedallion").getOrCreate()

In [0]:
# Configuração de catálogo, schemas e caminho da landing zone
catalog = "olist_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/landing"

print(f"catalog: {catalog}")
print(f"bronze_schema: {bronze_schema}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"Landing: {landing_path}")

catalog: olist_medallion
bronze_schema: olist_medallion.bronze
silver_schema: olist_medallion.silver
gold_schema: olist_medallion.gold
Landing: /Volumes/olist_medallion/bronze/landing


%md
### 1.2. Notebook — Camada Bronze

In [0]:
# Criação do catálogo, schemas e volume da landing zone
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.landing")

DataFrame[]

In [0]:
# Validação dos arquivos esperados na landing zone
expected_files = [
    "credits_and_tags_IMDB_TMDB.csv",
    "movies_financials_IMDB_TMDB.csv",
    "movies_info_TMDB_IMDB.csv",
    "movies_metrics_IMDB_TMDB.csv",
    "movies_reviews.csv"
]

try:
    existing = {f.name for f in dbutils.fs.ls(landing_path)}
except Exception as e:
    existing = set()
    print(f"[ALERTA] Não foi possível listar {landing_path}. Faça o upload dos arquivos antes de continuar.\n{e}")

missing = [f for f in expected_files if f not in existing]

if missing:
    print("[PENDENTE] Arquivos ainda não encontrados na landing zone:")
    for f in missing:
        print(f"  - {f}")
else:
    print("[OK] Todos os arquivos estão na landing zone.")
    for f in dbutils.fs.ls(landing_path):
        print(f"  - {f.name} ({f.size/1024:.1f} KB)")

[OK] Todos os arquivos estão na landing zone.
  - cotacao_dolar.csv (0.2 KB)
  - credits_and_tags_IMDB_TMDB.csv (21797.5 KB)
  - movies_financials_IMDB_TMDB.csv (1433.3 KB)
  - movies_info_TMDB_IMDB.csv (33129.0 KB)
  - movies_metrics_IMDB_TMDB.csv (3246.0 KB)
  - movies_reviews.csv (1878.4 KB)


### Ingestão Bronze

In [0]:
# Leitura dos CSVs da landing zone para DataFrames da camada Bronze
from pyspark.sql.functions import current_timestamp

path_credits_tags = f"{landing_path}/credits_and_tags_IMDB_TMDB.csv"
path_movies_financials = f"{landing_path}/movies_financials_IMDB_TMDB.csv"
path_movies_info = f"{landing_path}/movies_info_TMDB_IMDB.csv"
path_movies_metrics = f"{landing_path}/movies_metrics_IMDB_TMDB.csv"
path_movies_reviews = f"{landing_path}/movies_reviews.csv"

df_bronze_credits_tags_raw = spark.read.csv(path_credits_tags, header=True, inferSchema=True)
df_bronze_movies_financials_raw = spark.read.csv(path_movies_financials, header=True, inferSchema=True)
df_bronze_movies_info_raw = spark.read.csv(path_movies_info, header=True, inferSchema=True)
df_bronze_movies_metrics_raw = spark.read.csv(path_movies_metrics, header=True, inferSchema=True)
df_bronze_movies_reviews_raw = spark.read.csv(path_movies_reviews, header=True, inferSchema=True)

In [0]:
# Persistência das tabelas Bronze com timestamp de ingestão
(
    df_bronze_credits_tags_raw
    .withColumn("ingestion_datetime", current_timestamp())
    .write.format("Delta").mode("append")
    .saveAsTable(f"{catalog}.bronze.credits")
)

(
    df_bronze_movies_financials_raw
    .withColumn("ingestion_datetime", current_timestamp())
    .write.format("Delta").mode("append")
    .saveAsTable(f"{catalog}.bronze.movies_financials")
)

(
    df_bronze_movies_info_raw
    .withColumn("ingestion_datetime", current_timestamp())
    .write.format("Delta").mode("append")
    .saveAsTable(f"{catalog}.bronze.movies_info")
)

(
    df_bronze_movies_metrics_raw
    .withColumn("ingestion_datetime", current_timestamp())
    .write.format("Delta").mode("append")
    .saveAsTable(f"{catalog}.bronze.movies_metrics")
)

(
    df_bronze_movies_reviews_raw
    .withColumn("ingestion_datetime", current_timestamp())
    .write.format("Delta").mode("append")
    .saveAsTable(f"{catalog}.bronze.movies_reviews")
)

display(spark.table(f"{catalog}.bronze.credits").limit(5))

id,genres,production_companies,production_countries,spoken_languages,keywords,directors,writers,cast,ingestion_datetime
293660,"Action, Adventure, Comedy","20th Century Fox, The Donners' Company, Genre Films",United States of America,English,"superhero, anti hero, mercenary, based on comic, aftercreditsstinger, duringcreditsstinger",Tim Miller,"Rhett Reese, Paul Wernick","Ryan Reynolds, Morena Baccarin, Ed Skrein, T.J. Miller, Gina Carano, Leslie Uggams, Brianna Hildebrand, Stefan Kapičić, Karan Soni, Randal Reeder",2026-10-07T20:41:28.536Z
299536,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"English, Xhosa","sacrifice, magic, superhero, based on comic, space, battlefield, genocide, magical object, super power, aftercreditsstinger, marvel cinematic universe (mcu), cosmic","Anthony Russo, Joe Russo",N/A,"Robert Downey Jr., Chris Evans, Chris Hemsworth, Josh Brolin, Mark Ruffalo, Scarlett Johansson, Don Cheadle, Benedict Cumberbatch, Tom Holland, Chadwick Boseman",2026-10-07T20:41:28.536Z
299534,"Adventure, Science Fiction, Action",Marvel Studios,United States of America,"English, Japanese, Xhosa","superhero, time travel, space travel, time machine, based on comic, sequel, alien invasion, superhero team, marvel cinematic universe (mcu), alternate timeline, father daughter relationship, sister sister relationship","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Stan Lee, Jack Kirby, Joe Simon, Steve Englehart, Steve Gan, Bill Mantlo, Keith Giffen, Jim Starlin, Larry Lieber, Don Heck","Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris Hemsworth, Scarlett Johansson, Jeremy Renner, Josh Brolin, Don Cheadle, Paul Rudd, Benedict Cumberbatch",2026-10-07T20:41:28.536Z
475557,"Crime, Thriller, Drama","Warner Bros. Pictures, Joint Effort, Village Roadshow Pictures, Bron Studios, DC Films","Canada, United States of America",English,"dream, street gang, society, psychopath, clown, villain, based on comic, murder, psychological thriller, criminal mastermind, mental illness, anarchy, character study, clown makeup, subway train, social realism, supervillain, tv host, 1980s, mother son relationship, origin story, falling into madness, depressing",Todd Phillips,"Todd Phillips, Scott Silver, Bob Kane, Bill Finger, Jerry Robinson","Joaquin Phoenix, Robert De Niro, Zazie Beetz, Frances Conroy, Brett Cullen, Shea Whigham, Bill Camp, Glenn Fleshler, Leigh Gill, Josh Pais",2026-10-07T20:41:28.536Z
271110,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"Romanian, English, German, Russian","civil war, superhero, based on comic, sequel, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu), excited","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Joe Simon, Jack Kirby","Chris Evans, Robert Downey Jr., Scarlett Johansson, Sebastian Stan, Anthony Mackie, Don Cheadle, Jeremy Renner, Chadwick Boseman, Paul Bettany, Elizabeth Olsen",2026-10-07T20:41:28.536Z


### Ingestão de API

As próximas duas células de código que realizam a ingestão da API foram feitas com o auxílio de algoritmos compartilhados no grupo de Whatapp do RocketLab.

OBS: A princípio, os dados de cotação do dólar seriam obtidos diretamente pela API do Banco Central do Brasil (Olinda/SGS). Porém, devido a restrições do ambiente do Databricks, não foi possível realizar a consulta à API diretamente. Para dar continuidade ao desenvolvimento do projeto, foi utilizado um mockup, com os dados da cotação obtidos previamente e salvos em um arquivo CSV. Dessa forma, o arquivo cotacao_dolar.csv simula o resultado que seria retornado pela API e permite executar as etapas seguintes do código normalmente. O código foi modificado, de maneira que ao tentar buscar os dados pela API e falhar, o algoritmo fará a leitura a partir do mockup. 


In [0]:
# Widgets para definir o intervalo de datas da cotação do dólar
from datetime import date, timedelta

dbutils.widgets.text("data_inicio", (date.today() - timedelta(days=7)).strftime("%m-%d-%Y"), "Data início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", date.today().strftime("%m-%d-%Y"), "Data fim (MM-DD-AAAA)")

data_inicio_formatada = dbutils.widgets.get("data_inicio")
data_fim_formatada = dbutils.widgets.get("data_fim")

In [0]:
# Ingestão da cotação do dólar: tentativa via API do BCB com fallback para CSV
import requests
import pandas as pd

url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?"
    f"@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'"
    "&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

try:
    resposta = requests.get(url, timeout=30)
    resposta.raise_for_status()
    cotacoes = resposta.json()["value"]

    if not cotacoes:
        raise ValueError("A API não retornou cotações para o período informado. Ajuste as datas nos widgets.")

    df_cotacao_raw = spark.createDataFrame(pd.DataFrame(cotacoes))
    print("[OK] Cotações obtidas via API do BCB.")

except Exception as e:
    print(f"[AVISO] Falha ao consultar a API do BCB: {type(e).__name__}: {e}")
    print("Tentando carregar cotações a partir de arquivo CSV na landing zone...")

    csv_path = f"{landing_path}/cotacao_dolar.csv"
    try:
        df_cotacao_raw = spark.read.csv(csv_path, header=True, inferSchema=True)
        print(f"[OK] Cotações carregadas a partir de {csv_path}")
    except Exception as csv_e:
        raise RuntimeError(
            f"Não foi possível obter as cotações do dólar.\n"
            f"  - API do BCB falhou (sem acesso à internet no compute serverless).\n"
            f"  - Arquivo CSV não encontrado em {csv_path}.\n"
            f"Solução: baixe manualmente as cotações no site olinda.bcb.gov.br, "
            f"salve como 'cotacao_dolar.csv' e faça upload para a landing zone, "
            f"ou utilize um cluster clássico com acesso à internet."
        ) from csv_e

df_cotacao_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_cotacao_dolar")

# spark.sql(f"TRUNCATE TABLE {bronze_schema}.tb_cotacao_dolar")

# display(spark.table(f"{bronze_schema}.tb_cotacao_dolar").orderBy("dataHoraCotacao"))

display(
    spark.table(f"{bronze_schema}.tb_cotacao_dolar")
    .select(
        F.date_format("dataHoraCotacao", "MM-dd-yyyy").alias("dataHoraCotacao"),
        "cotacaoCompra",
        "ingestion_datetime"
    )
    .orderBy("dataHoraCotacao")
)

[AVISO] Falha ao consultar a API do BCB: ConnectionError: HTTPSConnectionPool(host='olinda.bcb.gov.br', port=443): Max retries exceeded with url: /olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='09-30-2026'&@dataFinalCotacao='10-07-2026'&$select=dataHoraCotacao,cotacaoCompra&$format=json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0xff8841255730>: Failed to resolve 'olinda.bcb.gov.br' ([Errno -3] Temporary failure in name resolution)"))
Tentando carregar cotações a partir de arquivo CSV na landing zone...
[OK] Cotações carregadas a partir de /Volumes/olist_medallion/bronze/landing/cotacao_dolar.csv


dataHoraCotacao,cotacaoCompra,ingestion_datetime
09-30-2026,5.1803,2026-10-07T21:29:58.497Z
10-01-2026,5.2073,2026-10-07T21:29:58.497Z
10-02-2026,5.2232,2026-10-07T21:29:58.497Z
10-05-2026,4.9853,2026-10-07T21:29:58.497Z
10-06-2026,4.9692,2026-10-07T21:29:58.497Z


# 1.3. Notebook — Camada Silver

Nenhuma alteração será realizada nas tabelas da camada Bronze. Todos os nomes de colunas devem estar em português
e a tipagem deve estar correta.


### 1) silver.tb_info_filmes (origem: tb_movies_info)

In [0]:
# Limpeza, renomeação de colunas para português e normalização do status
df_silver_movies_info = (
    df_bronze_movies_info_raw
    .dropDuplicates(["id"])
    .filter(F.col("id").isNotNull())
    .withColumn(
        "status",
         F.lower(
            F.trim(
                F.regexp_replace(
                    F.col("status"),
                        r"[-]+",
                        " "
                )
            )
        )
    )
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("title", "titulo")
    .withColumnRenamed("original_title", "titulo_original")
    .withColumnRenamed("original_language", "idioma_original")
    .withColumnRenamed("release_date", "data_lancamento")
    .withColumnRenamed("runtime", "duracao_minutos")
    .withColumnRenamed("status", "status_filme")
    .withColumnRenamed("overview", "sinopse")
    .withColumnRenamed("tagline", "frase_divulgacao")
)

df_silver_movies_info = (
    df_silver_movies_info
    .withColumn(
        "status_filme",
        F.when(
            F.col("status_filme") == "released",
            "Lançado"
        )
        .when(
            F.col("status_filme") == "post production",
            "Pós-Produção"
        )
        .when(
            F.col("status_filme") == "in production",
            "Em Produção"
        )
        .when(
            F.col("status_filme") == "planned",
            "Planejado"
        )
        .when(
            F.col("status_filme") == "rumored",
            "Rumores"
        )
        .when(
            F.col("status_filme") == "canceled",
            "Cancelado"
        )
        .otherwise("Não Informado")
    )
)

df_silver_movies_info = (
    df_silver_movies_info
    .withColumn(
        "data_lancamento",
        F.coalesce(
            F.try_to_date(F.trim(F.col("data_lancamento")), F.lit("yyyy-MM-dd")),
            F.try_to_date(F.trim(F.col("data_lancamento")), F.lit("dd/MM/yyyy")),
            F.try_to_date(F.trim(F.col("data_lancamento")), F.lit("yyyy/MM/dd"))
        )
    )
)

df_silver_movies_info = df_silver_movies_info.withColumn("ano_lancamento", F.year(F.col("data_lancamento")))

# Escrita na tabela Silver e visualização comparativa com a Bronze
df_silver_movies_info.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.movie_info")

display(df_silver_movies_info.orderBy("id_filme").limit(10))
display(df_bronze_movies_info_raw.orderBy("id").limit(10))

id_filme,tconst,titulo,titulo_original,idioma_original,data_lancamento,duracao_minutos,status_filme,sinopse,frase_divulgacao,ano_lancamento
14564,tt0498381,Rings,Rings,en,2017-02-01,102,Lançado,"\Julia becomes worried about her boyfriend Holt when he explores the dark urban legend of a mysterious videotape said to kill the watcher seven days after viewing. She sacrifices herself to save her boyfriend and in doing so makes a horrifying discovery: there is a """"\""""movie within the movie""""\"""" that no one has ever seen before.""""""""First you watch it. Then you die.",null,2017
32471,tt1587420,Mixtape,Mixtape,en,2021-12-03,94,Lançado,null,null,2021
38258,tt0093119,Grizzly II: Revenge,Grizzly II: Revenge,en,2020-02-17,74,Lançado,\All hell breaks loose when a giant grizzly,reacting to the slaughter of her cubs by poachers,2020
38492,tt0215569,Billy Joel - Live at Yankee Stadium,Billy Joel Live at Yankee Stadium,en,2022-06-22,86,Lançado,Billy Joel plays his greatest hits in the Big Apple.,null,2022
38700,tt1502397,Bad Boys for Life,Bad Boys for Life,en,2020-01-15,124,Lançado,"Marcus and Mike are forced to confront new threats, career changes, and midlife crises as they join the newly created elite team AMMO of the Miami police department to take down the ruthless Armando Armas, the vicious leader of a Miami drug cartel.",Ride together. Die together.,2020
42018,tt0090907,The Horse Thief,盗马贼,zh,2019-03-19,88,Lançado,"Devout Buddhists, Norbu and Dolma live with their young son Tashi in a clan in Tibet. Norbu is a highwayman. After Norbu is charged with stealing from the temple, he and his family are banished. Impoverished and marginalized, they can do little when their beloved son becomes ill. Tashi dies of a fever. After a second son is born, Norbu focuses his every action on keeping this child alive, seeking re-admission to the clan for his wife and child, then risking all to save them from isolation and starvation in winter.",null,2019
42330,tt10443316,Monkey Magic,大闹西游,zh,2018-09-22,66,Lançado,"Dearth Voyd, the ruler of the dark side of the universe, wants to turn the human world into a place of violence and evil!! However, a huge meteor falls upon Flower-Fruit mountain giving birth to Kongo, a monkey made of stone! Kongo, now sets out on his quest to fight evil and restore order to a desperate land. Episodes 1-3",null,2018
43074,tt1289401,Ghostbusters,Ghostbusters,en,2016-07-14,117,Lançado,"Following a ghost invasion of Manhattan, paranormal enthusiasts Erin Gilbert and Abby Yates, nuclear engineer Jillian Holtzmann, and subway worker Patty Tolan band together to stop the otherworldly threat.",Who You Gonna Call?,2016
45033,tt1202339,20 Seconds of Joy,20 Seconds of Joy,de,2018-01-01,60,Lançado,"Traces the story of an extreme athlete, past and present; but also explores the psychology behind life, death, risk and the confrontation of fear.",null,2018
46983,tt0051471,The Song of Styrene,Le Chant du styrène,fr,2022-05-23,13,Lançado,Le chant du Styrène is a 1958 French documentary film directed by Alain Resnais. The film was an order by French industrial group Pechiney to highlight the merits of plastics.,null,2022


id,tconst,title,original_title,original_language,release_date,runtime,status,overview,tagline
14564,tt0498381,Rings,Rings,en,2017-02-01,102,released,"\Julia becomes worried about her boyfriend Holt when he explores the dark urban legend of a mysterious videotape said to kill the watcher seven days after viewing. She sacrifices herself to save her boyfriend and in doing so makes a horrifying discovery: there is a """"\""""movie within the movie""""\"""" that no one has ever seen before.""""""""First you watch it. Then you die.",null
32471,tt1587420,Mixtape,Mixtape,en,2021-12-03,94,Released,null,null
38258,tt0093119,Grizzly II: Revenge,Grizzly II: Revenge,en,2020-02-17,74,Released,\All hell breaks loose when a giant grizzly,reacting to the slaughter of her cubs by poachers
38492,tt0215569,Billy Joel - Live at Yankee Stadium,Billy Joel Live at Yankee Stadium,en,2022-06-22,86,Released,Billy Joel plays his greatest hits in the Big Apple.,null
38700,tt1502397,Bad Boys for Life,Bad Boys for Life,en,2020-01-15,124,Released,"Marcus and Mike are forced to confront new threats, career changes, and midlife crises as they join the newly created elite team AMMO of the Miami police department to take down the ruthless Armando Armas, the vicious leader of a Miami drug cartel.",Ride together. Die together.
42018,tt0090907,The Horse Thief,盗马贼,zh,2019-03-19,88,Released,"Devout Buddhists, Norbu and Dolma live with their young son Tashi in a clan in Tibet. Norbu is a highwayman. After Norbu is charged with stealing from the temple, he and his family are banished. Impoverished and marginalized, they can do little when their beloved son becomes ill. Tashi dies of a fever. After a second son is born, Norbu focuses his every action on keeping this child alive, seeking re-admission to the clan for his wife and child, then risking all to save them from isolation and starvation in winter.",null
42330,tt10443316,Monkey Magic,大闹西游,zh,2018-09-22,66,Released,"Dearth Voyd, the ruler of the dark side of the universe, wants to turn the human world into a place of violence and evil!! However, a huge meteor falls upon Flower-Fruit mountain giving birth to Kongo, a monkey made of stone! Kongo, now sets out on his quest to fight evil and restore order to a desperate land. Episodes 1-3",null
43074,tt1289401,Ghostbusters,Ghostbusters,en,2016-07-14,117,Released,"Following a ghost invasion of Manhattan, paranormal enthusiasts Erin Gilbert and Abby Yates, nuclear engineer Jillian Holtzmann, and subway worker Patty Tolan band together to stop the otherworldly threat.",Who You Gonna Call?
45033,tt1202339,20 Seconds of Joy,20 Seconds of Joy,de,2018-01-01,60,Released,"Traces the story of an extreme athlete, past and present; but also explores the psychology behind life, death, risk and the confrontation of fear.",null
46983,tt0051471,The Song of Styrene,Le Chant du styrène,fr,23/05/2022,13,Released,Le chant du Styrène is a 1958 French documentary film directed by Alain Resnais. The film was an order by French industrial group Pechiney to highlight the merits of plastics.,null


### 2) silver.tb_financeiro_filmes (origem: tb_movies_financials)

In [0]:
from pyspark.sql.types import DecimalType

df_silver_movies_financials = (
    df_bronze_movies_financials_raw
    .dropDuplicates(["id"])
    .filter(F.col("id").isNotNull())
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("budget", "orcamento_usd")
    .withColumnRenamed("revenue", "receita_usd")
    .withColumnRenamed("orcamento_usd", "orcamento_brl")
    .withColumnRenamed("receita_usd", "receita_brl")
)

# ==========================================
# 1. Tratar valores textuais como NULL
# ==========================================

df_silver_movies_financials = df_silver_movies_financials.withColumn(
    "orcamento_brl",
    F.when(
        F.lower(F.trim(F.col("orcamento_brl"))).isin(
            "unknown",
            "não informado",
            "nao informado",
            "",
            "null",
            "n/a"
        ),
        None
    ).otherwise(F.trim(F.col("orcamento_brl")))
)

df_silver_movies_financials = df_silver_movies_financials.withColumn(
    "receita_brl",
    F.when(
        F.lower(F.trim(F.col("receita_brl"))).isin(
            "unknown",
            "não informado",
            "nao informado",
            "",
            "null",
            "n/a"
        ),
        None
    ).otherwise(F.trim(F.col("receita_brl")))
)

# ==========================================
# 2. Remover símbolos e separadores
# ==========================================

df_silver_movies_financials = df_silver_movies_financials.withColumn(
    "orcamento_brl",
    F.regexp_replace(
        F.col("orcamento_brl"),
        r"[$€£R$\s,.]",
        ""
    )
)

df_silver_movies_financials = df_silver_movies_financials.withColumn(
    "receita_brl",
    F.regexp_replace(
        F.col("receita_brl"),
        r"[$€£R$\s,.]",
        ""
    )
)

# ==========================================
# 3. Converter para Decimal
# ==========================================

df_silver_movies_financials = df_silver_movies_financials.withColumn(
    "orcamento_brl",
    F.col("orcamento_brl").try_cast(DecimalType(18, 2))
)

df_silver_movies_financials = df_silver_movies_financials.withColumn(
    "receita_brl",
    F.col("receita_brl").try_cast(DecimalType(18, 2))
)

# ==========================================
# 4. Zeros e negativos → NULL
# ==========================================

df_silver_movies_financials = df_silver_movies_financials.withColumn(
    "orcamento_brl",
    F.when(
        F.col("orcamento_brl") > 0,
        F.col("orcamento_brl")
    )
)

df_silver_movies_financials = df_silver_movies_financials.withColumn(
    "receita_brl",
    F.when(
        F.col("receita_brl") > 0,
        F.col("receita_brl")
    )
)

# ==========================================
# 5. Conversão USD → BRL
# ==========================================

# Cotação atual do dólar/real
cotacao_usd_brl = 5.40

df_silver_movies_financials = df_silver_movies_financials.withColumn(
    "orcamento_brl",
    F.col("orcamento_brl") * F.lit(cotacao_usd_brl)
)

df_silver_movies_financials = df_silver_movies_financials.withColumn(
    "receita_brl",
    F.col("receita_brl") * F.lit(cotacao_usd_brl)
)

# ==========================================
# 6. Lucro USD
# ==========================================

df_silver_movies_financials = df_silver_movies_financials.withColumn(
    "lucro_usd",
    F.when(
        F.col("receita_brl").isNotNull() &
        F.col("orcamento_brl").isNotNull(),
        F.col("receita_brl") - F.col("orcamento_brl")
    )
)

# ==========================================
# 7. Lucro BRL
# ==========================================

df_silver_movies_financials = df_silver_movies_financials.withColumn(
    "lucro_brl",
    F.when(
        F.col("receita_brl").isNotNull() &
        F.col("orcamento_brl").isNotNull(),
        F.col("receita_brl") - F.col("orcamento_brl")
    )
)

# ==========================================
# 8. Margem de lucro (%)
# ==========================================

df_silver_movies_financials = df_silver_movies_financials.withColumn(
    "margem_lucro_percentual",
    F.when(
        F.col("orcamento_brl").isNotNull() &
        F.col("receita_brl").isNotNull() &
        (F.col("orcamento_brl") > 0),
        (
            (F.col("receita_brl") - F.col("orcamento_brl"))
            / F.col("orcamento_brl")
        ) * 100
    )
)

df_silver_movies_financials.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.movie_financials")

display(df_silver_movies_financials.orderBy("id_filme").limit(10))
display(df_bronze_movies_financials_raw.orderBy("id").limit(10))

id_filme,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_percentual
14564,1.35E8,4.48636806E8,3.13636806E8,3.13636806E8,232.32356
32471,null,null,null,null,null
38258,null,null,null,null,null
38492,null,null,null,null,null
38700,null,2.3031283176000004E9,null,null,null
42018,null,null,null,null,null
42330,null,null,null,null,null
43074,7.776E8,1.2373965486000001E9,4.5979654860000014E8,4.5979654860000014E8,59.13021458333335
45033,1820880.0000000002,null,null,null,null
46983,null,null,null,null,null


id,budget,revenue
14564,25000000,83080890
32471,0,0
38258,7.5M,0
38492,0,0
38700,90.0M,426505244
42018,0,0
42330,0,0
43074,144000000,229147509
45033,337200,0
46983,0,0


### 3) silver.tb_metricas_engajamento (origem: tb_movies_metrics)


In [0]:
df_silver_movies_metrics = (
    df_bronze_movies_metrics_raw
    .dropDuplicates(["id"])
    .filter(F.col("id").isNotNull())
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("popularity", "popularidade")
    .withColumnRenamed("vote_average", "nota_media_tmdb")
    .withColumnRenamed("averageRating", "nota_media_imdb")
    .withColumnRenamed("vote_count", "qtd_votos_tmdb")
    .withColumnRenamed("numVotes", "qtd_votos_imdb")
)

# ==========================================
# 1. Popularidade
# ==========================================

# Trata textos de ausência e remove espaços
df_silver_movies_metrics = df_silver_movies_metrics.withColumn(
    "popularidade",
    F.when(
        F.lower(F.trim(F.col("popularidade"))).isin(
            "unknown",
            "não informado",
            "nao informado",
            "null",
            "n/a",
            ""
        ),
        None
    ).otherwise(
        F.trim(F.col("popularidade"))
    )
)

# Normaliza vírgula como separador decimal
df_silver_movies_metrics = df_silver_movies_metrics.withColumn(
    "popularidade",
    F.when(
        F.col("popularidade").contains(",") &
        ~F.col("popularidade").contains("."),
        F.regexp_replace(
            F.col("popularidade"),
            ",",
            "."
        )
    )
    .otherwise(
        F.col("popularidade")
    )
)

# Conversão segura:
# textos como "and was not officially released until 2020"
# serão convertidos para NULL
df_silver_movies_metrics = df_silver_movies_metrics.withColumn(
    "popularidade",
    F.expr("try_cast(popularidade AS DOUBLE)")
)

# Valores negativos são inválidos
df_silver_movies_metrics = df_silver_movies_metrics.withColumn(
    "popularidade",
    F.when(
        F.col("popularidade") >= 0,
        F.col("popularidade")
    )
)

# ==========================================
# 2. Nota média TMDB
# ==========================================

df_silver_movies_metrics = df_silver_movies_metrics.withColumn(
    "nota_media_tmdb",
    F.expr("try_cast(trim(nota_media_tmdb) AS DOUBLE)")
)

df_silver_movies_metrics = df_silver_movies_metrics.withColumn(
    "nota_media_tmdb",
    F.when(
        (F.col("nota_media_tmdb") >= 0) &
        (F.col("nota_media_tmdb") <= 10),
        F.col("nota_media_tmdb")
    )
)


# ==========================================
# 3. Nota média IMDb
# ==========================================

df_silver_movies_metrics = df_silver_movies_metrics.withColumn(
    "nota_media_imdb",
    F.expr("try_cast(trim(nota_media_imdb) AS DOUBLE)")
)

df_silver_movies_metrics = df_silver_movies_metrics.withColumn(
    "nota_media_imdb",
    F.when(
        (F.col("nota_media_imdb") >= 0) &
        (F.col("nota_media_imdb") <= 10),
        F.col("nota_media_imdb")
    )
)


# ==========================================
# 4. Quantidade de votos TMDB
# ==========================================

df_silver_movies_metrics = df_silver_movies_metrics.withColumn(
    "qtd_votos_tmdb",
    F.expr("try_cast(trim(qtd_votos_tmdb) AS BIGINT)")
)

df_silver_movies_metrics = df_silver_movies_metrics.withColumn(
    "qtd_votos_tmdb",
    F.when(
        F.col("qtd_votos_tmdb") >= 0,
        F.col("qtd_votos_tmdb")
    )
)


# ==========================================
# 5. Quantidade de votos IMDb
# ==========================================

df_silver_movies_metrics = df_silver_movies_metrics.withColumn(
    "qtd_votos_imdb",
    F.expr("try_cast(trim(qtd_votos_imdb) AS BIGINT)")
)

df_silver_movies_metrics = df_silver_movies_metrics.withColumn(
    "qtd_votos_imdb",
    F.when(
        F.col("qtd_votos_imdb") >= 0,
        F.col("qtd_votos_imdb")
    )
)

df_silver_movies_metrics.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.movie_metrics")

display(df_silver_movies_metrics.orderBy("id_filme").limit(10))
display(df_bronze_movies_metrics_raw.orderBy("id").limit(10))

id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
14564,24.584,4.966,2375,null,46286
32471,8.929,7.064,118,6.6,4617
38258,null,3.161,28,null,null
38492,2.98,7.1,null,7.8,212
38700,46.619,7.139,7570,6.5,199420
42018,3.403,6.63,27,6.8,1750
42330,1.561,10.0,1,6.4,34
43074,40.052,5.371,5976,null,260417
45033,0.6,8.0,2,null,154
46983,0.693,8.5,2,7.0,1209


id,popularity,vote_average,vote_count,averageRating,numVotes
14564,24.584,4.966,2375,null,46286
32471,8.929,7.064,118,6.6,4617
38258,and was not officially released until 2020,3.161,28,André Szöts,"Joan McCall, David Sheldon"
38492,2.98,7.1,null,7.8,212
38700,"46,619",7.139,7570,6.5,199420
42018,3.403,6.63,27,6.8,1750
42330,1.561,10.0,1,6.4,34
43074,40.052,5.371,5976,null,260417
45033,0.6,8.0,2,null,154
46983,0.693,8.5,2,7.0,1209


### 4) silver.tb_avaliacoes_usuarios (origem: tb_movies_reviews)

In [0]:
# ==========================================
# 1. Remover registros integralmente duplicados
# ==========================================

df_silver_movies_reviews = (
    df_bronze_movies_reviews_raw
    .filter(F.col("id").isNotNull())
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("nome", "nome_usuario")
    .withColumnRenamed("nota", "nota_usuario")
    .withColumnRenamed("comentario", "comentario_usuario")
)

df_silver_movies_reviews = df_silver_movies_reviews.dropDuplicates([
    "id_filme",
    "nome_usuario",
    "nota_usuario",
    "comentario_usuario"
])


# ==========================================
# 2. Tratar e validar a nota do usuário
# ==========================================

# Conversão segura para número
df_silver_movies_reviews = df_silver_movies_reviews.withColumn(
    "nota_usuario",
    F.expr("try_cast(trim(nota_usuario) AS DOUBLE)")
)

# Notas fora do intervalo 0-10 -> NULL
df_silver_movies_reviews = df_silver_movies_reviews.withColumn(
    "nota_usuario",
    F.when(
        (F.col("nota_usuario") >= 0) &
        (F.col("nota_usuario") <= 10),
        F.col("nota_usuario")
    )
)


# ==========================================
# 3. Tratar comentários vazios ou com espaços
# ==========================================

df_silver_movies_reviews = df_silver_movies_reviews.withColumn(
    "comentario_usuario",
    F.when(
        F.col("comentario_usuario").isNull() |
        (F.trim(F.col("comentario_usuario")) == ""),
        "Sem comentário"
    ).otherwise(
        F.col("comentario_usuario")
    )
)

df_silver_movies_reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.movie_reviews")

display(df_silver_movies_reviews.orderBy("id_filme").limit(10))
display(df_bronze_movies_reviews_raw.orderBy("id").limit(10))

id_filme,nome_usuario,nota_usuario,comentario_usuario
32471,Eduardo Barbosa 494,8.3,"Ótimo entretenimento, não decepciona."
38258,Maria Lima 308,0.9,Sem comentário
38258,Carolina Monteiro,9.4,Sem comentário
38492,Mônica Fernandes 243,2.5,Sem comentário
38700,Leandro Moreira 157,0.1,"Terrível, um dos piores que já vi."
46983,Marcio Santos 147,0.2,Não recomendo de jeito nenhum.
58431,Fábio Correia 159,4.7,"Decepcionante, esperava muito mais."
66213,Juliana Gomes 626,4.9,"Ruim, não vale o tempo investido."
76812,Fernanda Silva 471,9.3,Perfeito! Um dos melhores filmes que já vi.
80765,Thiago Monteiro 238,1.2,"Terrível, um dos piores que já vi."


id,nome,nota,comentario
32471,Eduardo Barbosa 494,8.3,"Ótimo entretenimento, não decepciona."
38258,Carolina Monteiro,9.4,null
38258,Maria Lima 308,0.9,null
38492,Mônica Fernandes 243,2.5,null
38700,Leandro Moreira 157,0.1,"Terrível, um dos piores que já vi."
46983,Marcio Santos 147,0.2,Não recomendo de jeito nenhum.
58431,Fábio Correia 159,4.7,"Decepcionante, esperava muito mais."
66213,Juliana Gomes 626,4.9,"Ruim, não vale o tempo investido."
76812,Fernanda Silva 471,9.3,Perfeito! Um dos melhores filmes que já vi.
80765,Thiago Monteiro 238,1.2,"Terrível, um dos piores que já vi."


### 5) silver.tb_generos (origem: tb_credits_and_tags, coluna genres)

In [0]:
# ============================================================
# 1. SILVER - GÊNEROS
# ============================================================

df_silver_generos = (
    df_bronze_credits_tags_raw

    .withColumn(
        "generos",
        F.regexp_replace(
            F.col("genres"),
            r"\s*;\s*",
            ","
        )
    )

    .withColumn(
        "genero",
        F.explode(
            F.split(
                F.col("generos"),
                r"\s*,\s*"
            )
        )
    )

    .withColumn(
        "genero",
        F.trim(F.col("genero"))
    )

    .filter(
        F.col("genero").isNotNull() &
        (F.col("genero") != "")
    )

    .filter(
        ~F.col("genero").rlike(r"^\d+([.,]\d+)?$")
    )

    .filter(
        F.col("genero").isin(
            "Action",
            "Adventure",
            "Animation",
            "Comedy",
            "Crime",
            "Documentary",
            "Drama",
            "Family",
            "Fantasy",
            "History",
            "Horror",
            "Music",
            "Mystery",
            "Romance",
            "Science Fiction",
            "TV Movie",
            "Thriller",
            "War",
            "Western"
        )
    )

    .dropDuplicates([
        "id",
        "genero"
    ])

    .select(
        F.col("id").alias("id_filme"),
        "genero"
    )
)

df_silver_generos.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.generos")

display(
    df_silver_generos
    .orderBy("id_filme")
    .limit(10)
)

# ============================================================
# 2. SILVER - EMPRESAS DE PRODUÇÃO
# ============================================================

df_silver_empresas = (
    df_bronze_credits_tags_raw

    .withColumn(
        "empresas_producao",
        F.regexp_replace(
            F.col("production_companies"),
            r"\s*;\s*",
            ","
        )
    )

    .withColumn(
        "empresa_producao",
        F.explode(
            F.split(
                F.col("empresas_producao"),
                r"\s*,\s*"
            )
        )
    )

    .withColumn(
        "empresa_producao",
        F.trim(F.col("empresa_producao"))
    )

    .filter(
        F.col("empresa_producao").isNotNull() &
        (F.col("empresa_producao") != "")
    )

    .filter(
        ~F.col("empresa_producao").rlike(r"^\d+([.,]\d+)?$")
    )

    .dropDuplicates([
        "id",
        "empresa_producao"
    ])

    .select(
        F.col("id").alias("id_filme"),
        "empresa_producao"
    )
)

df_silver_empresas.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.empresas")

display(
    df_silver_empresas
    .orderBy("id_filme")
    .limit(10)
)

# ============================================================
# 3. SILVER - PAÍSES DE PRODUÇÃO
# ============================================================

df_silver_paises = (
    df_bronze_credits_tags_raw

    .withColumn(
        "paises_producao",
        F.regexp_replace(
            F.col("production_countries"),
            r"\s*;\s*",
            ","
        )
    )

    .withColumn(
        "pais_producao",
        F.explode(
            F.split(
                F.col("paises_producao"),
                r"\s*,\s*"
            )
        )
    )

    .withColumn(
        "pais_producao",
        F.trim(F.col("pais_producao"))
    )

    .filter(
        F.col("pais_producao").isNotNull() &
        (F.col("pais_producao") != "")
    )

    .filter(
        ~F.col("pais_producao").rlike(r"^\d+([.,]\d+)?$")
    )

    .dropDuplicates([
        "id",
        "pais_producao"
    ])

    .select(
        F.col("id").alias("id_filme"),
        "pais_producao"
    )
)

df_silver_paises.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.paises")

display(
    df_silver_paises
    .orderBy("id_filme")
    .limit(10)
)

# ============================================================
# 4. SILVER - IDIOMAS FALADOS
# ============================================================

df_silver_idiomas = (
    df_bronze_credits_tags_raw

    .withColumn(
        "idiomas_falados",
        F.regexp_replace(
            F.col("spoken_languages"),
            r"\s*;\s*",
            ","
        )
    )

    .withColumn(
        "idioma_falado",
        F.explode(
            F.split(
                F.col("idiomas_falados"),
                r"\s*,\s*"
            )
        )
    )

    .withColumn(
        "idioma_falado",
        F.trim(F.col("idioma_falado"))
    )

    .filter(
        F.col("idioma_falado").isNotNull() &
        (F.col("idioma_falado") != "")
    )

    .filter(
        ~F.col("idioma_falado").rlike(r"^\d+([.,]\d+)?$")
    )

    .dropDuplicates([
        "id",
        "idioma_falado"
    ])

    .select(
        F.col("id").alias("id_filme"),
        "idioma_falado"
    )
)

df_silver_idiomas.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.idiomas")

display(
    df_silver_idiomas
    .orderBy("id_filme")
    .limit(10)
)


# ============================================================
# 5. SILVER - PALAVRAS-CHAVE
# ============================================================

df_silver_keywords = (
    df_bronze_credits_tags_raw

    .withColumn(
        "palavras_chave",
        F.regexp_replace(
            F.col("keywords"),
            r"\s*;\s*",
            ","
        )
    )

    .withColumn(
        "palavra_chave",
        F.explode(
            F.split(
                F.col("palavras_chave"),
                r"\s*,\s*"
            )
        )
    )

    .withColumn(
        "palavra_chave",
        F.trim(F.col("palavra_chave"))
    )

    .filter(
        F.col("palavra_chave").isNotNull() &
        (F.col("palavra_chave") != "")
    )

    .filter(
        ~F.col("palavra_chave").rlike(r"^\d+([.,]\d+)?$")
    )

    .dropDuplicates([
        "id",
        "palavra_chave"
    ])

    .select(
        F.col("id").alias("id_filme"),
        "palavra_chave"
    )
)

df_silver_keywords.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.keywords")

display(
    df_silver_keywords
    .orderBy("id_filme")
    .limit(10)
)

# ============================================================
# 6. SILVER - DIRETORES
# ============================================================

df_silver_diretores = (
    df_bronze_credits_tags_raw

    .withColumn(
        "diretores",
        F.regexp_replace(
            F.col("directors"),
            r"\s*;\s*",
            ","
        )
    )

    .withColumn(
        "diretor",
        F.explode(
            F.split(
                F.col("diretores"),
                r"\s*,\s*"
            )
        )
    )

    .withColumn(
        "diretor",
        F.trim(F.col("diretor"))
    )

    .filter(
        F.col("diretor").isNotNull() &
        (F.col("diretor") != "")
    )

    .filter(
        ~F.col("diretor").rlike(r"^\d+([.,]\d+)?$")
    )

    .dropDuplicates([
        "id",
        "diretor"
    ])

    .select(
        F.col("id").alias("id_filme"),
        "diretor"
    )
)

df_silver_diretores.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.diretores")

display(
    df_silver_diretores
    .orderBy("id_filme")
    .limit(10)
)

# ============================================================
# 7. SILVER - ROTEIRISTAS
# ============================================================

df_silver_roteiristas = (
    df_bronze_credits_tags_raw

    .withColumn(
        "roteiristas",
        F.regexp_replace(
            F.col("writers"),
            r"\s*;\s*",
            ","
        )
    )

    .withColumn(
        "roteirista",
        F.explode(
            F.split(
                F.col("roteiristas"),
                r"\s*,\s*"
            )
        )
    )

    .withColumn(
        "roteirista",
        F.trim(F.col("roteirista"))
    )

    .filter(
        F.col("roteirista").isNotNull() &
        (F.col("roteirista") != "")
    )

    .filter(
        ~F.col("roteirista").rlike(r"^\d+([.,]\d+)?$")
    )

    .dropDuplicates([
        "id",
        "roteirista"
    ])

    .select(
        F.col("id").alias("id_filme"),
        "roteirista"
    )
)

df_silver_roteiristas.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.roteiristas")

display(
    df_silver_roteiristas
    .orderBy("id_filme")
    .limit(10)
)

# ============================================================
# 8. SILVER - ELENCO
# ============================================================

df_silver_elenco = (
    df_bronze_credits_tags_raw

    .withColumn(
        "elenco",
        F.regexp_replace(
            F.col("cast"),
            r"\s*;\s*",
            ","
        )
    )

    .withColumn(
        "ator",
        F.explode(
            F.split(
                F.col("elenco"),
                r"\s*,\s*"
            )
        )
    )

    .withColumn(
        "ator",
        F.trim(F.col("ator"))
    )

    .filter(
        F.col("ator").isNotNull() &
        (F.col("ator") != "")
    )

    .filter(
        ~F.col("ator").rlike(r"^\d+([.,]\d+)?$")
    )

    .dropDuplicates([
        "id",
        "ator"
    ])

    .select(
        F.col("id").alias("id_filme"),
        "ator"
    )
)

df_silver_elenco.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.elenco")

display(
    df_silver_elenco
    .orderBy("id_filme")
    .limit(10)
)

id_filme,genero
14564,Horror
32471,Family
32471,Comedy
38492,Music
38700,Action
38700,Crime
38700,Thriller
42018,Drama
42330,Adventure
42330,Fantasy


id_filme,empresa_producao
14564,Paramount
14564,Macari/Edelstein
14564,Parkes+MacDonald Image Nation
14564,Waddieish Claretrap
14564,Vertigo Entertainment
32471,Jim Wedaa Productions
32471,Maven Screen Media
38258,The Stage Is Set... And The Dinner Is Served.
38492,Columbia Music Video
38700,Don Simpson/Jerry Bruckheimer Films


id_filme,pais_producao
14564,United States of America
32471,United States of America
38258,Horror
38258,Music
38258,Thriller
38492,United States of America
38700,United States of America
42018,China
42330,China
43074,United States of America


id_filme,idioma_falado
14564,English
32471,English
38258,Harlequin Pictures
38258,GBGB International
38492,English
38700,English
38700,Spanish
42018,Mandarin
42018,Tibetan
42330,Mandarin


id_filme,palavra_chave
14564,maggot
14564,seven days
14564,vhs
14564,hallucination
14564,video tape
14564,supernatural
14564,investigation
14564,sequel
14564,death
14564,curse


id_filme,diretor
14564,F. Javier Gutiérrez
32471,Valerie Weiss
38258,English
38492,Jon Small
38700,Bilall Fallah
38700,Adil El Arbi
42018,Peicheng Pan
42018,Zhuangzhuang Tian
43074,Paul Feig
45033,Jens Hoffmann


id_filme,roteirista
14564,Akiva Goldsman
14564,David Loucka
14564,Kôji Suzuki
14564,Jacob Estes
32471,Stacey Menear
38258,sequel
38258,wild animal
38258,bear
38258,concert
38258,bodily dismemberment


id_filme,ator
14564,Zach Roerig
14564,Matilda Lutz
14564,Johnny Galecki
14564,Vincent D'Onofrio
14564,Chuck David Willis
14564,Patrick Walker
14564,Aimee Teegarden
14564,Bonnie Morgan
14564,Alex Roe
14564,Laura Slade Wiggins


### 6) silver.tb_pessoas_empresas (origem: tb_credits_and_tags, colunas cast, directors, writers, production_companies)


In [0]:
df_silver_movies_credits_tag = (
    df_bronze_credits_tags_raw.select("cast", "directors", "writers", "production_companies")
    .withColumnRenamed("cast", "Ator")
    .withColumnRenamed("directors", "Diretor")
    .withColumnRenamed("writers", "Roteirista")
    .withColumnRenamed("production_companies", "Produtora")
)

# ============================================================
# DIMENSÃO UNIFICADA DE ENTIDADES
#
# Origem:
#   df_silver_movies_credits_tag
#
# Colunas de origem:
#   Ator
#   Diretor
#   Roteirista
#   Produtora
#
# Resultado:
#   entidade
#   tipo_entidade
# ============================================================


# ============================================================
# 1. FUNÇÃO DE LIMPEZA DAS ENTIDADES
# ============================================================

def limpar_entidade(df, coluna, tipo):

    return (
        df

        # ----------------------------------------------------
        # Seleciona a coluna da entidade
        # ----------------------------------------------------
        .select(
            F.col(coluna).alias("entidade")
        )

        # ----------------------------------------------------
        # Remove espaços no início e no final
        # ----------------------------------------------------
        .withColumn(
            "entidade",
            F.trim(F.col("entidade"))
        )

        # ----------------------------------------------------
        # Remove nulos e valores vazios
        # ----------------------------------------------------
        .filter(
            F.col("entidade").isNotNull() &
            (F.col("entidade") != "")
        )

        # ----------------------------------------------------
        # Remove valores puramente numéricos
        #
        # Exemplos:
        # 123
        # 0.6
        # 0,6
        # 123,
        # 123,456
        # 123.456,
        # -10.5
        # ----------------------------------------------------
        .filter(
            ~F.col("entidade").rlike(
                r"^\s*[-+]?\d+([.,]\d+)*\s*,?\s*$"
            )
        )

        # ----------------------------------------------------
        # Remove arquivos de imagem
        #
        # Exemplos:
        # imagem.jpg
        # poster.png
        # ator.jpeg
        # foto.webp
        # ----------------------------------------------------
        .filter(
            ~F.lower(F.col("entidade")).rlike(
                r"\.(jpg|jpeg|png|gif|webp|bmp|tiff?)\s*,?\s*$"
            )
        )

        # ----------------------------------------------------
        # Remove caminhos de arquivos
        #
        # Exemplos:
        # /images/ator.jpg
        # images/ator.jpg
        # C:\images\ator.jpg
        # ----------------------------------------------------
        .filter(
            ~F.col("entidade").rlike(
                r"[/\\]"
            )
        )

        # ----------------------------------------------------
        # Remove valores contendo aspas
        #
        # Exemplos:
        # "A Broken Life"""
        # "A Horror Styled"
        # "A Mixture Of Tradition..."
        # ----------------------------------------------------
        .filter(
            ~F.col("entidade").rlike(
                r"""["']"""
            )
        )

        # ----------------------------------------------------
        # Remove textos excessivamente longos
        # ----------------------------------------------------
        .filter(
            F.length(F.col("entidade")) <= 100
        )

        # ----------------------------------------------------
        # Remove textos que apresentam características de
        # frases descritivas deslocadas da origem.
        #
        # Exemplos:
        # doesn't
        # don't
        # isn't
        # weren't
        # this
        # that
        # love
        # take
        # took
        # see
        # etc.
        # ----------------------------------------------------
        .filter(
            ~F.lower(F.col("entidade")).rlike(
                r"\b("
                r"doesn't|don't|isn't|aren't|wasn't|weren't|"
                r"does|do|is|are|was|were|has|have|had|"
                r"this|that|these|those|"
                r"take|takes|took|"
                r"love|loves|loved|"
                r"see|sees|saw"
                r")\b"
            )
        )

        # ----------------------------------------------------
        # Padroniza capitalização
        #
        # TOM HANKS -> Tom Hanks
        # tom hanks -> Tom Hanks
        # Tom Hanks -> Tom Hanks
        # ----------------------------------------------------
        .withColumn(
            "entidade",
            F.initcap(
                F.lower(
                    F.col("entidade")
                )
            )
        )

        # ----------------------------------------------------
        # Adiciona o tipo da entidade
        # ----------------------------------------------------
        .withColumn(
            "tipo_entidade",
            F.lit(tipo)
        )
    )


# ============================================================
# 2. ATOR
# ============================================================

df_atores = limpar_entidade(
    df_silver_movies_credits_tag,
    "Ator",
    "Ator"
)


# ============================================================
# 3. DIRETOR
# ============================================================

df_diretores = limpar_entidade(
    df_silver_movies_credits_tag,
    "Diretor",
    "Diretor"
)


# ============================================================
# 4. ROTEIRISTA
# ============================================================

df_roteiristas = limpar_entidade(
    df_silver_movies_credits_tag,
    "Roteirista",
    "Roteirista"
)


# ============================================================
# 5. PRODUTORA
# ============================================================

df_produtoras = limpar_entidade(
    df_silver_movies_credits_tag,
    "Produtora",
    "Produtora"
)


# ============================================================
# 6. UNIFICAÇÃO DAS QUATRO CATEGORIAS
# ============================================================

df_dim_entidades = (
    df_atores
    .unionByName(df_diretores)
    .unionByName(df_roteiristas)
    .unionByName(df_produtoras)
)


# ============================================================
# 7. REMOVER DUPLICIDADES
#
# A combinação entidade + tipo_entidade deve ser única.
# ============================================================

df_dim_entidades = (
    df_dim_entidades
    .dropDuplicates([
        "entidade",
        "tipo_entidade"
    ])
)


# ============================================================
# 8. ORDENAÇÃO
# ============================================================

df_dim_entidades = (
    df_dim_entidades
    .orderBy(
        "tipo_entidade",
        "entidade"
    )
)


# ============================================================
# 9. SALVAR COMO TABELA DELTA
# ============================================================

df_dim_entidades.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        f"{silver_schema}.dim_entidades"
    )


# ============================================================
# 10. VISUALIZAR RESULTADO
# ============================================================

display(
    df_dim_entidades.limit(50)
)

entidade,tipo_entidade
"2 Chainz, Bankroll Fresh, Skooly",Ator
"2 Pretty Beebaby, Jerald Andrews, Matthew Lee Spidell Blaque",Ator
"50 Cent, Eminem, Snoop Dogg, Mary J. Blige, Dr. Dre, Kendrick Lamar",Ator
"6 Keys, Lil Baby, Djbj, Snap Dogg, Duck & Shorty, Lou Gram, Cifton Mcgarity Iv, Mox",Ator
6ix9ine,Ator
"A Leslie Kies, Victor Manso, Eric Hailey, Julia Farino",Ator
A Putz. But Right Now,Ator
A Shadow In The Dark Productions,Ator
A. Michelle Harleston,Ator
A.c. Thompson,Ator


### 7) silver.tb_cotacao_dolar (origem: tb_cotacao_dolar)

In [0]:
# ============================================================
# 1. PREPARAR A TABELA DE COTAÇÃO
# ============================================================

df_cotacao = (
    df_cotacao_raw

    # Converte a data/hora para DATE
    .withColumn(
        "data_cotacao",
        F.to_date(F.col("dataHoracotacao"))
    )

    # Converte a cotação para decimal
    .withColumn(
        "cotacao_compra",
        F.col("cotacaoCompra").cast("decimal(18,6)")
    )

    # Mantém somente os campos necessários
    .select(
        "data_cotacao",
        "cotacao_compra"
    )

    # Caso exista mais de uma cotação no mesmo dia,
    # mantém uma única ocorrência
    .groupBy("data_cotacao")
    .agg(
        F.last("cotacao_compra", ignorenulls=True)
        .alias("cotacao_compra")
    )
)


# ============================================================
# 2. IDENTIFICAR O INTERVALO DO HISTÓRICO
# ============================================================

df_periodo = (
    df_cotacao
    .agg(
        F.min("data_cotacao").alias("data_inicial"),
        F.max("data_cotacao").alias("data_final")
    )
)


# ============================================================
# 3. CRIAR CALENDÁRIO DIÁRIO CONTÍNUO
# ============================================================

df_calendario = (
    df_periodo

    .select(
        F.explode(
            F.sequence(
                F.col("data_inicial"),
                F.col("data_final"),
                F.expr("INTERVAL 1 DAY")
            )
        ).alias("data_cotacao")
    )
)


# ============================================================
# 4. ASSOCIAR AS COTAÇÕES AO CALENDÁRIO
# ============================================================

df_cotacao_continua = (
    df_calendario

    .join(
        df_cotacao,
        on="data_cotacao",
        how="left"
    )
)


# ============================================================
# 5. FORWARD FILL
#
# Para cada dia sem cotação, busca a última cotação
# disponível anteriormente.
# ============================================================

window_forward_fill = (
    Window
    .orderBy("data_cotacao")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)


df_cotacao_continua = (
    df_cotacao_continua

    .withColumn(
        "cotacao_compra",
        F.last(
            "cotacao_compra",
            ignorenulls=True
        ).over(window_forward_fill)
    )
)


# ============================================================
# 6. GARANTIR O TIPO DECIMAL
# ============================================================

df_cotacao_continua = (
    df_cotacao_continua
    .withColumn(
        "cotacao_compra",
        F.col("cotacao_compra").cast("decimal(18,6)")
    )
)


# ============================================================
# 7. SELECIONAR AS COLUNAS FINAIS
# ============================================================

df_cotacao_continua = (
    df_cotacao_continua
    .select(
        "data_cotacao",
        "cotacao_compra"
    )
    .orderBy("data_cotacao")
)


# ============================================================
# 8. SALVAR COMO TABELA DELTA
# ============================================================

df_cotacao_continua.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        f"{silver_schema}.cotacao_dolar"
    )


# ============================================================
# 9. VISUALIZAR RESULTADO
# ============================================================

display(
    df_cotacao_continua.limit(50)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


data_cotacao,cotacao_compra
2026-09-30,5.180300
2026-10-01,5.207300
2026-10-02,5.223200
2026-10-03,5.223200
2026-10-04,5.223200
2026-10-05,4.985300
2026-10-06,4.969200


# Parte 2 — Consagração do Negócio (Camada Gold)
Nesta etapa você construirá os Data Marts consolidados, a modelagem dimensional (Star Schema) para consumo do
time de BI, e a tabela de contexto para o time de IA. Utilize exibições de DataFrames em tela (display()) para responder
às perguntas de negócio solicitadas. Nenhuma biblioteca de visualização de gráficos é necessária.


### Entrega 1 — Modelagem Dimensional (Star Schema)